# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Shehzadi434/flyrank-Internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


**Lane:** Lane 1 — Content Refresh Prediction

**ML Task Type:** Binary Classification

**Why:** Predict whether a page is declining (Yes/No). This directly maps to the action: review or don't review. The baseline in Notebook 01 used classification, so results are comparable.

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. Ready to Go.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. Ready to Go.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

label_counts = df["trend_direction"].value_counts()
print("Label distribution:")
print(label_counts)
print(f"\nDeclining rate: {label_counts.get('down', 0) / len(df):.1%}")


Label distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining rate: 54.2%


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining_label` — derived from `trend_direction == "down"`

**Source:** Observed data — impressions dropped >20% in last 30 days vs previous 30 days.

**Why it works:** Based on real measurements, not product decisions. Matches Notebook 01 baseline for fair comparison.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Declining (1): {df['is_declining_label'].sum():,} rows")
print(f"Not declining (0): {(df['is_declining_label'] == 0).sum():,} rows")
print(f"Declining rate: {df['is_declining_label'].mean():.1%}")

df[["content_id", "trend_direction", "impressions_90d", "is_declining_label"]].head(3)


Declining (1): 16,262 rows
Not declining (0): 13,738 rows
Declining rate: 54.2%


,content_id,trend_direction,impressions_90d,is_declining_label
0,content_304f48230142,down,3803,1
1,content_a1fb4e703a9e,down,15320,1
2,content_9aa793d4d895,down,12581,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*


**Primary Metric:** Precision@50

**Why:** Matches real decision — content teams review a fixed number of pages (top 50). Measures: "Of top 50 flagged, how many are actually declining?"

**Target:** ≥ 0.740 (match Random Forest from Notebook 01)

**Baseline:** 0.240 (hand-written rule)

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import json

try:
    with open("outputs/model_results.json", "r") as f:
        results = json.load(f)
    baseline = results["baseline"]["baseline_precision_at_50"]
    rf = results["models"]["random_forest"]["precision_at_50"]
    print(f"Baseline (hand rule): {baseline:.3f}")
    print(f"Random Forest: {rf:.3f}")
    print(f"ML beats rule by {rf/baseline:.1f}x")
    print(f"\nTop 50: Baseline = {int(baseline*50)}/50, ML = {int(rf*50)}/50 correct")
except:
    print("Run Notebook 01 first. Expected: baseline 0.240, RF 0.740")


Run Notebook 01 first. Expected: baseline 0.240, RF 0.740


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. The unit of analysis, as a real dataframe

**Unit:** One row = one page (content item)

**Why:** Decision is page-level — "Should we review THIS page?" Each page has unique features (impressions, CTR, position). Matches how content teams think about inventory.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(f"Total pages: {len(df):,}")
print("One row = one page\n")
df[["content_id", "client_id", "impressions_90d", "ctr", "avg_position", "trend_direction"]].head(5)


Total pages: 30,000
One row = one page



,content_id,client_id,impressions_90d,ctr,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,0.76,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,15320,0.05,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,0.09,36.5,down
3,content_331d6c4de07b,client_19581e27de,11751,0.49,6.2,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,0.13,44.0,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why fixed rules fail:**
- Simple thresholds ignore context (position, volume, content type)
- Feature interactions are complex — hand rules can't capture them all
- Different clients have different patterns

**Evidence:** Notebook 01 showed Random Forest (0.740 Precision@50) beats hand rule (0.240) by 3.1×.

**ML Advantage:** Learns feature interactions automatically, adapts to data, handles many features, and ranks pages so humans prioritize effectively.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

features = ["search_volume", "ctr", "avg_position", "impressions_90d", "word_count"]
correlations = {}
for f in features:
    valid = df[df[f].notna()]
    if len(valid) > 0:
        correlations[f] = valid[f].corr(valid["is_declining_label"])

print("Correlation with declining label:")
for f, c in correlations.items():
    print(f"  {f}: {c:.3f}")

print("\nNo single feature strongly predicts decline.")
print("ML combines multiple weak signals to beat simple rules.")


Correlation with declining label:
  search_volume: -0.019
  ctr: -0.062
  avg_position: -0.029
  impressions_90d: -0.018
  word_count: 0.090

No single feature strongly predicts decline.
ML combines multiple weak signals to beat simple rules.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.